#Config et importations des données

In [ ]:
# installation de Pyspark et de l'API Kaggle
!pip install pyspark
!pip install kaggle

In [ ]:
#configuration de la clé API de kaggle
#from genericpath import exists
from google.colab import files
import os

if not os.path.exists('/root/.Kaggle/kaggle.json'):
  print("Uploader le fichier Kaggle.json")
  files.upload()

  !mkdir -p ~/.kaggle
  !cp kaggle.json ~/.kaggle/
  !chmod 600 ~/.kaggle/kaggle.json  #pour les permission de lecture et d'écriture
  !rm kaggle.json

Uploader le fichier Kaggle.json


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
rm: cannot remove 'kaggle.json': No such file or directory


In [ ]:
#Téléchargement et desippage du dataset olist
!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -o brazilian-ecommerce.zip -d olist_data

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:00<00:00, 110MB/s]

Archive:  brazilian-ecommerce.zip
  inflating: olist_data/olist_customers_dataset.csv  
  inflating: olist_data/olist_geolocation_dataset.csv  
  inflating: olist_data/olist_order_items_dataset.csv  
  inflating: olist_data/olist_order_payments_dataset.csv  
  inflating: olist_data/olist_order_reviews_dataset.csv  
  inflating: olist_data/olist_orders_dataset.csv  
  inflating: olist_data/olist_products_dataset.csv  
  inflating: olist_data/olist_sellers_dataset.csv  
  inflating: olist_data/product_category_name_translation.csv  


In [ ]:
#Initialisation de spark session
from  pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
        .appName("Olist_Intensive_Training") \
        .config("spark.sql.shuffle.partitions", "8") \
        .getOrCreate()

In [ ]:
#Chargement automatique de tous les dataset dans un dictionaire
datasets = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

dfs = {}
for dataset in datasets:
  name = dataset.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
  path = f"/content/olist_data/{dataset}"
  dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
  print(f"Dataset '{name}' chargé. Nombre de lignes : {dfs[name].count()}")

Dataset 'customers' chargé. Nombre de lignes : 99441
Dataset 'geolocation' chargé. Nombre de lignes : 1000163
Dataset 'order_items' chargé. Nombre de lignes : 112650
Dataset 'order_payments' chargé. Nombre de lignes : 103886
Dataset 'order_reviews' chargé. Nombre de lignes : 104162
Dataset 'orders' chargé. Nombre de lignes : 99441
Dataset 'products' chargé. Nombre de lignes : 32951
Dataset 'sellers' chargé. Nombre de lignes : 3095
Dataset 'product_category_name_translation' chargé. Nombre de lignes : 71


In [ ]:
#test
dfs["orders"].show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

#Nettoyage et ingestion

Objectif : Maîtriser le chargement, le renommage et le traitement des valeurs manquantes.

In [ ]:
df_customer = dfs["customers"]
#df_customer.show()

#0-verification si customer_id et customer_unique_id sont unique
from pyspark.sql.window import Window
from pyspark.sql.functions import countDistinct

total_lignes = df_customer.count()

verif = (
    df_customer
        .agg(
            countDistinct("customer_id").alias("customer_id_count"),
            countDistinct("customer_unique_id").alias("customer_unique_id_count")
        )
        .withColumn("total_ligne_count", lit(total_lignes))
)

verif.show()


+-----------------+------------------------+-----------------+
|customer_id_count|customer_unique_id_count|total_ligne_count|
+-----------------+------------------------+-----------------+
|            99441|                   96096|            99441|
+-----------------+------------------------+-----------------+



In [ ]:
#1-séquence incrtementale d'entiers

df_customer = (
    df_customer
    .withColumn("customer_id_seq", row_number().over(Window.orderBy("customer_id")))
)

df_customer.show()

+--------------------+--------------------+------------------------+--------------------+--------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|customer_id_seq|
+--------------------+--------------------+------------------------+--------------------+--------------+---------------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                    6273|              osasco|            SP|              1|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|         itapecerica|            MG|              2|
|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830|        nova venecia|            ES|              3|
|0002414f953443074...|4893ad4ea28b2c5b3...|                   39664|            mendonca|            MG|              4|
|000379cdec6255224...|0b83f73b19c2019e1...|                    4841|           sao paulo|            SP|              5|
|0004164d20a9e969a...|104bdb7e6a

In [ ]:
#2-filtrez les commandes pour ne garder que celles dont le statut est delivered

delivered_orders = (
    dfs["orders"]
    .filter(col("order_status") == "delivered")
)

not_delivered_orders = (
    dfs["orders"]
    .filter(col("order_status") != "delivered")
)

#delivered_orders.show()


In [ ]:
#3- Suppression des ligne dont la date d'appeobation des nulle

approved_delivered_orders = (
    delivered_orders.filter(col("order_approved_at").isNotNull())
)
#approved_delivered_orders.count()
#delivered_orders.count()

In [ ]:
#Nbre de clients inique dans le dataset customers

customer_number = df_customer.select("customer_id").distinct().count()
print(f"Nombre de clients uniques dans le dataset customers : {customer_number}")

Nombre de clients uniques dans le dataset customers : 99441


In [ ]:
#5- le nombre de codes postaux distincts dans la table geolocation

distinct_postal_codes = dfs["geolocation"].select("geolocation_zip_code_prefix").distinct().count()
print(f"Nombre de codes postaux distincts dans la table geolocation : {distinct_postal_codes}")

Nombre de codes postaux distincts dans la table geolocation : 19015


In [ ]:
#6-convertions des colonnes de poids et de dimensions en type Float

df_products = (
    dfs["products"]
    .withColumn("product_weight_g", col("product_weight_g").cast("float"))
    .withColumn("product_length_cm", col("product_length_cm").cast("float"))
    .withColumn("product_height_cm", col("product_height_cm").cast("float"))
    .withColumn("product_width_cm", col("product_width_cm").cast("float"))
)
df_products.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|           225.0|             16.0|             10.0|            14.0|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|          1000.0|             30.0|             18.0|            20.0|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [ ]:
#df_products.printSchema()

In [ ]:
#7- Categorisation des produits

df_products_with_categ = (
    df_products
    .withColumn("product_category", when(
        col("product_weight_g") < 5000, "Léger"
    ).when(
        (col("product_weight_g") >= 5000) & (col("product_weight_g") <= 10000), "Moyen"
    ).when(
        (col("product_weight_g") > 10000) & (col("product_weight_g") <= 20000), "Lourd"
    ).otherwise("SuperLourd")
    )
)

df_products_with_categ.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|           225.0|             16.0|             10.0|            14.0|           Léger|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|          1000.0|             30.0|             18.0|            20.0|           Léger|
|96bd76ec8

In [ ]:
#8- les 5 mots clés les plus fréquent dans la catégories produits

must_frq_words = (
    df_products_with_categ
    .select(explode(split(col("product_category_name"), " ")).alias("mot_frq"))
    .groupBy("mot_frq")
    .count()
    .orderBy(col("count").desc())
    .limit(5)
)

must_frq_words.show()

+--------------------+-----+
|             mot_frq|count|
+--------------------+-----+
|     cama_mesa_banho| 3029|
|       esporte_lazer| 2867|
|    moveis_decoracao| 2657|
|        beleza_saude| 2444|
|utilidades_domest...| 2335|
+--------------------+-----+



In [ ]:
# 9-Affichez les 10 produits les plus volumineux (longueur * largeur * hauteur). vérifiez s'ils sont les plus lourds

most_volume_products = (
    df_products_with_categ
    .withColumn("volume", col("product_length_cm") * col("product_height_cm") * col("product_width_cm"))
    .orderBy(col("volume").desc())
    .limit(10)
)

most_volume_products.show()


+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+--------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category|  volume|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+--------+
|256a9c364b75753b9...| utilidades_domest...|                 59|                       649|                 1|         25250.0|             68.0|             66.0|            66.0|      SuperLourd|296208.0|
|0b48eade13cfad433...|     moveis_decoracao|                 28|                       489|                 1|         14000.0|             70.0|             60.0|         

In [ ]:
#10-Vérification qu'aucune valeur de prix dans order_items n'est négative.

df_order_items = dfs["order_items"]
print("le nbre de valeurs prix négative dans order_items : ", df_order_items.filter(col("price") < 0).count())

le nbre de valeurs prix négative dans order_items :  0


#Les Jointures et Agrégations

Objectif : Créer une "Table de Faits" pour analyser la performance des vendeurs.

In [ ]:
#1-Jointure entre les tables order_items, orders et sellers

df_order_items = dfs["order_items"]
df_sellers = dfs["sellers"]
df_orders = dfs["orders"]

df_orders_sellers = (
    approved_delivered_orders
    .join(df_order_items, on="order_id", how="left")
    .join(df_sellers, on="seller_id", how="left")
    .select(
        "seller_id", "order_id", "customer_id", "order_status", "price" ,"order_purchase_timestamp",
        "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date", "product_id",
        "shipping_limit_date", "freight_value", "seller_zip_code_prefix", "seller_city", "seller_state"
    )
)

#df_orders_sellers.show()
df_orders_sellers.write.mode("overwrite").format("parquet").save("/content/tables/df_orders_sellers")

In [ ]:
#2-le chiffre d'affaires total et le nbre de commende par vendeur

CA_nbre_cmd = (
    df_orders_sellers
    .groupBy("seller_id")
    .agg(
        round(sum("price"),2).alias("CA_total"),
        countDistinct("order_id").alias("nbre_cmd")
    )
)

#CA_nbre_cmd.show()

In [ ]:
#3-Ajoute d'une colonne calculant le panier moyen par vendeur

CA_nbre_cmd = (
    CA_nbre_cmd
    .withColumn("panier_moyen", round(col("CA_total") / col("nbre_cmd"),2))
)

CA_nbre_cmd.show()

+--------------------+---------+--------+------------+
|           seller_id| CA_total|nbre_cmd|panier_moyen|
+--------------------+---------+--------+------------+
|7c67e1448b00f6e96...|186570.05|     973|      191.75|
|b33e7c55446eabf8f...| 44215.17|     149|      296.75|
|5dceca129747e92ff...|111126.73|     322|      345.11|
|17a053fcb14bd2195...| 40353.24|     124|      325.43|
|48efc9d94a9834137...|    345.6|      33|       10.47|
|98dac6635aee4995d...|  4115.75|      89|       46.24|
|850f4f8af5ea87287...| 45285.37|     186|      243.47|
|77530e9772f57a62c...|  45676.6|     312|       146.4|
|643214e62b870443c...|  3229.23|      65|       49.68|
|8a32e327fe2c1b351...|  6101.12|     126|       48.42|
|276677b5d08786d5d...|   3424.2|      31|      110.46|
|2138ccb85b11a4ec1...| 12089.22|     388|       31.16|
|1835b56ce799e6a4d...| 33005.32|     417|       79.15|
|0ea22c1cfbdc755f8...|  10592.1|     234|       45.27|
|8e6d7754bc7e0f22c...| 14107.38|      84|      167.95|
|ff063b022

In [ ]:
# 4-Triez par CA décroissant et sauvegardez le résultat au format Parquet.

CA_nbre_cmd = (
    CA_nbre_cmd
    .orderBy(col("CA_total").desc())
)

CA_nbre_cmd.write.mode("overwrite").format("parquet").save("/content/tables/CA_nbre_cmd")

In [ ]:
CA_nbre_cmd.show()

+--------------------+---------+--------+------------+
|           seller_id| CA_total|nbre_cmd|panier_moyen|
+--------------------+---------+--------+------------+
|4869f7a5dfa277a7d...|226987.93|    1124|      201.95|
|53243585a1d6dc264...|217940.44|     348|      626.27|
|4a3ca9315b744ce9f...|196882.12|    1772|      111.11|
|fa1c13f2614d7b5c4...|190917.14|     578|      330.31|
|7c67e1448b00f6e96...|186570.05|     973|      191.75|
|7e93a43ef30c4f03f...|165981.49|     319|      520.32|
|da8622b14eb17ae28...|159816.87|    1311|       121.9|
|7a67c85e85bb2ce85...|139418.72|    1142|      122.08|
|1025f0e2d44d7041d...|138208.56|     910|      151.88|
|955fee9216a65b617...|131836.71|    1261|      104.55|
|46dc3b2cc0980fb8e...|122811.38|     503|      244.16|
|6560211a19b47992c...|120702.83|    1819|       66.36|
|620c87c171fb2a6dd...| 112461.5|     722|      155.76|
|7d13fca1522535862...|112436.18|     558|       201.5|
|5dceca129747e92ff...|111126.73|     322|      345.11|
|1f50f9201

In [ ]:
#df_orders.printSchema()

In [ ]:
#5-Calculez le temps réel de livraison (delivered_customer_date - order_purchase_timestamp) pour chaque commande.

df_orders_sellers = (
    df_orders_sellers
    .withColumn("temps_livraison", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
)

df_orders_sellers.show()

+--------------------+--------------------+--------------------+------------+-----+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+-------------+----------------------+--------------------+------------+---------------+
|           seller_id|            order_id|         customer_id|order_status|price|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|          product_id|shipping_limit_date|freight_value|seller_zip_code_prefix|         seller_city|seller_state|temps_livraison|
+--------------------+--------------------+--------------------+------------+-----+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+-------------+----------------------+-------------

In [ ]:
#6-Créez une colonne is_late booléenne renseignant s'il y a eu du retard par rapport à la date de livraison estimée.

df_orders_sellers = (
    df_orders_sellers
    .withColumn("is_late", col("order_delivered_customer_date") > col("order_estimated_delivery_date"))
)

df_orders_sellers.show()

+--------------------+--------------------+--------------------+------------+-----+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+-------------+----------------------+--------------------+------------+---------------+-------+
|           seller_id|            order_id|         customer_id|order_status|price|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|          product_id|shipping_limit_date|freight_value|seller_zip_code_prefix|         seller_city|seller_state|temps_livraison|is_late|
+--------------------+--------------------+--------------------+------------+-----+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+-------------+--------------------

In [ ]:
nb_true = df_orders_sellers.where(col("is_late")== True).count()
print(nb_true)

8714


In [ ]:
#df_geolocalisation = dfs["geolocation"]
df_geolocalisation = (
    dfs["geolocation"]
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("lat"),
        avg("geolocation_lng").alias("lng")
    )
)

In [ ]:
# Localisation des vendeurs
df_orders_sellers_localisation = (
    df_orders_sellers.alias("ord_sel")
    .join(df_geolocalisation.alias("geo_s"), col("geo_s.geolocation_zip_code_prefix") == col("ord_sel.seller_zip_code_prefix"), "inner")
)

# Localisation des clients
df_customer_localisation = (
    df_customer.alias("cust")
    .join(df_geolocalisation.alias("geo_c"),col("geo_c.geolocation_zip_code_prefix") == col("cust.customer_zip_code_prefix"),"inner")
)

# Jointure vendeurs + clients
df_seller_customer_geolocalisation = (
    df_orders_sellers_localisation.alias("sel_loc")
    .join(df_customer_localisation.alias("cust_loc"), on="customer_id", how="inner")
    .select(
        "sel_loc.customer_id", "sel_loc.seller_id", "sel_loc.order_id", "sel_loc.product_id", "sel_loc.order_status", "sel_loc.price", "sel_loc.order_purchase_timestamp", "sel_loc.order_approved_at",
        "sel_loc.order_delivered_carrier_date", "sel_loc.order_delivered_customer_date", "sel_loc.order_estimated_delivery_date", "sel_loc.shipping_limit_date", "sel_loc.freight_value",
        "sel_loc.seller_zip_code_prefix", "sel_loc.seller_city", "sel_loc.seller_state", "sel_loc.temps_livraison", "sel_loc.is_late", "cust_loc.customer_unique_id",
        "cust_loc.customer_zip_code_prefix", "cust_loc.customer_city", "cust_loc.customer_state", "cust_loc.customer_id_seq",
        col("sel_loc.geolocation_zip_code_prefix").alias("seller_geolocation_zip_code_prefix"),
        col("sel_loc.lat").alias("lat_seller"),
        col("sel_loc.lng").alias("lng_seller"),
        col("cust_loc.lat").alias("lat_customer"),
        col("cust_loc.lng").alias("lng_customer")
    )
)

df_seller_customer_geolocalisation.show()

+--------------------+--------------------+--------------------+--------------------+------------+------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+----------------------+--------------+------------+---------------+-------+--------------------+------------------------+--------------------+--------------+---------------+----------------------------------+-------------------+-------------------+-------------------+-------------------+
|         customer_id|           seller_id|            order_id|          product_id|order_status| price|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|shipping_limit_date|freight_value|seller_zip_code_prefix|   seller_city|seller_state|temps_livraison|is_late|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|customer

In [ ]:
#df_sellers.show()
#delaie livraison moyen par ville

df_temps_livraison_par_ville = (
    df_orders_sellers
    .groupBy("seller_city")
    .agg(avg(col("temps_livraison")))
)


from pyspark.sql.functions import col, sqrt, pow
from math import pi

coef = pi * 6352 / 180

df_seller_customer_geolocalisation = (
    df_seller_customer_geolocalisation
    .withColumn(
        "distance",
        sqrt(pow((col("lat_seller") - col("lat_customer")) * coef, 2) + pow((col("lng_seller") - col("lng_customer")) * coef, 2))
    )
    .orderBy(col("distance").desc())
)
df_seller_customer_geolocalisation.show()

#corelation entre delais de livraison et la distance entre le vendeur et le client


#verifier si parmi les commendes en retard il y a un sur-representation de certaines ville  (pour les clients ou les vendeurs)



+--------------------+--------------------+--------------------+--------------------+------------+------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+-------------+----------------------+--------------------+------------+---------------+-------+--------------------+------------------------+--------------------+--------------+---------------+----------------------------------+-------------------+-------------------+--------------------+-------------------+------------------+
|         customer_id|           seller_id|            order_id|          product_id|order_status| price|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|shipping_limit_date|freight_value|seller_zip_code_prefix|         seller_city|seller_state|temps_livraison|is_late|  customer_unique_id|customer_zip_code_prefix|       custo

In [ ]:
print(dfs["order_reviews"].count(),dfs["orders"].count())

104162 99441


In [ ]:
#7- jointure entre les tables orders et order_reviews
from pyspark.sql.functions import broadcast
import time

# Méthode 1
start_time1 = time.time()
orders_and_orders_reviews1 = (
    dfs["orders"].join(dfs["order_reviews"], on="order_id", how="inner")
)
orders_and_orders_reviews1.count()
end_time1 = time.time()
print(f"Temps écoulé méthode 1 : {end_time1 - start_time1}")

# Méthode 2
start_time2 = time.time()
orders_and_orders_reviews2 = (
    dfs["order_reviews"].join(broadcast(dfs["orders"]), on="order_id", how="inner")
)
orders_and_orders_reviews2.count()
end_time2 = time.time()
print(f"Temps écoulé méthode 2 : {end_time2 - start_time2}")

Temps écoulé méthode 1 : 2.42749285697937
Temps écoulé méthode 2 : 2.0200047492980957


In [ ]:
#orders_and_orders_reviews2.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|           review_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|73fc7af87114b3971...|7bc2406110b926393...|           4|  

In [ ]:
from pyspark.sql.functions import expr
#8. Calculez le nombre de commandes passées pour chaque mois de chaque année et évaluez l'évolution.
from pyspark.sql.functions import year, month, count

nbre_commende_annee_mois = (
    orders_and_orders_reviews2
    .withColumn("annee", year("order_purchase_timestamp"))
    .withColumn("mois", month("order_purchase_timestamp"))
    .groupBy("annee", "mois")
    .agg(count("order_id").alias("nbre_commande"))
    .orderBy("annee", "mois")
)
nbre_commende_annee_mois.show()

+-----+----+-------------+
|annee|mois|nbre_commande|
+-----+----+-------------+
| 2016|   9|            4|
| 2016|  10|          321|
| 2016|  12|            1|
| 2017|   1|          797|
| 2017|   2|         1776|
| 2017|   3|         2676|
| 2017|   4|         2394|
| 2017|   5|         3703|
| 2017|   6|         3250|
| 2017|   7|         4032|
| 2017|   8|         4341|
| 2017|   9|         4277|
| 2017|  10|         4626|
| 2017|  11|         7534|
| 2017|  12|         5638|
| 2018|   1|         7245|
| 2018|   2|         6758|
| 2018|   3|         7187|
| 2018|   4|         6894|
| 2018|   5|         6848|
+-----+----+-------------+
only showing top 20 rows


In [ ]:
#9-les 5 villes où les frais de port moyens sont les plus élevés.

top_5_ville = (
    df_orders_sellers
    .groupBy(col("seller_city"))
    .agg(avg("freight_value").alias("avg_freight"))
    .orderBy(col("avg_freight").desc())
    .limit(5)
)

top_5_ville.show()


+--------------------+------------------+
|         seller_city|       avg_freight|
+--------------------+------------------+
|          lages - sc|168.53333333333333|
|sao francisco do sul|            150.22|
|          california|           143.775|
|sao  jose dos pin...|142.39999999999998|
|         nova trento|            131.85|
+--------------------+------------------+



In [ ]:
#10-Remplacement des valeurs manquantes dans les commentaires de reviews par la chaîne "Pas de commentaire".

#from pyspark.sql.functions import col, coalesce, lit

df_reviews = (
    dfs["order_reviews"]
    .withColumn(
        "review_comment_title",
        coalesce(col("review_comment_title"), lit("no comments"))
    )
    .withColumn(
        "review_comment_message",
        coalesce(col("review_comment_message"), lit("Pas de commentaire"))
    )
)

df_reviews.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|         no comments|    Pas de commentaire| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|         no comments|    Pas de commentaire| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|         no comments|    Pas de commentaire| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|         no comments|  Recebi bem antes ...| 2017-04-21 00:00:00|   

#Fenêtrage et Analyse Temporelle

Objectif : Analyser la fidélité client avec les Window Functions.

In [ ]:
#1-intervalle de temps par clients

commandes_clients = df_orders_sellers.join(df_customer, on="customer_id", how="inner")

windowSpec = (
    Window.partitionBy("customer_unique_id")
    .orderBy("order_purchase_timestamp")
)

customer_interval = (
    commandes_clients
    .withColumn(
        "num_custumer_id", row_number().over(windowSpec)
        )
    .withColumn(
        "date_precedent", lag("order_purchase_timestamp",1).over(windowSpec)
    )
    .withColumn(
        "jours_entre_commandes", datediff("order_purchase_timestamp", "date_precedent")
        )
      .select(
          "customer_id","order_purchase_timestamp", "date_precedent", "jours_entre_commandes"
      )
      .filter(col("num_custumer_id")==2)
      .orderBy(col("jours_entre_commandes").desc())

)

customer_interval.show()

+--------------------+------------------------+-------------------+---------------------+
|         customer_id|order_purchase_timestamp|     date_precedent|jours_entre_commandes|
+--------------------+------------------------+-------------------+---------------------+
|3b6828a50ffe54694...|     2018-06-07 19:03:12|2016-10-06 19:33:34|                  609|
|6d334aaab9f45b27e...|     2018-05-09 13:49:19|2016-10-05 21:10:56|                  581|
|6efe2f2c813379b61...|     2018-05-04 11:14:37|2016-10-08 18:45:34|                  573|
|7022a8efc480a54d4...|     2018-08-25 11:01:56|2017-03-19 08:36:36|                  524|
|bb465058eeef2697a...|     2018-03-13 22:28:21|2016-10-05 12:32:55|                  524|
|9c159f3aa83cd57f7...|     2018-07-24 16:44:34|2017-02-17 18:27:58|                  522|
|0881e1574aa6b621c...|     2018-06-25 12:06:07|2017-01-26 23:48:56|                  515|
|9e36e87b2ed55c78b...|     2018-07-14 19:38:26|2017-02-15 12:56:00|                  514|
|960ffb45a

In [ ]:
#commandes_clients.printSchema()

In [ ]:
CA_par_customer = (
    commandes_clients
    .groupBy("customer_id")
    .agg(sum(col("price")).alias("Ciffre_affaire"))
    .orderBy(col("Ciffre_affaire").desc())
).show()

+--------------------+--------------+
|         customer_id|Ciffre_affaire|
+--------------------+--------------+
|1617b1357756262bf...|       13440.0|
|ec5b2ba62e5743423...|        7160.0|
|c6e2731c5b391845f...|        6735.0|
|f48d464a0baaea338...|        6729.0|
|3fd6777bbce08a352...|        6499.0|
|05455dfa7cd02f13d...|        5934.6|
|df55c14d1476a9a34...|        4799.0|
|24bbf5fd2f2e1b359...|        4690.0|
|3d979689f636322c6...|        4590.0|
|cc803a2c412833101...|        4400.0|
|1afc82cd60e303ef0...|       4399.87|
|35a413c7ca3c69756...|       4099.99|
|e9b0d0eb3015ef1c9...|        4059.0|
|c6695e3b1e48680db...|        3999.9|
|926b6a6fb8b6081e0...|        3999.0|
|3be2c536886b2ea46...|        3980.0|
|31e83c01fce824d0f...|        3930.0|
|eb7a157e8da9c488c...|        3899.0|
|19b32919fa1198aef...|        3700.0|
|66657bf1753d82d0a...|       3699.99|
+--------------------+--------------+
only showing top 20 rows


In [ ]:
#les clients qui representye les 80% du chiffre d'affaire total
CA_par_customer = (
    commandes_clients
    .groupBy("customer_id")
    .agg(sum(col("price")).alias("Ciffre_affaire"))
    .orderBy(col("Ciffre_affaire").desc())
)

total_ca = CA_par_customer.agg(sum("Ciffre_affaire").alias("total")).collect()[0]["total"]

print(total_ca)

13219827.68001749


In [ ]:
#2- les clients qui representye les 80% du chiffre d'affaire total

CA_par_customer = (
    commandes_clients
    .groupBy("customer_unique_id")
    .agg(sum(col("price")).alias("chiffre_affaire"))
    .orderBy(col("chiffre_affaire").desc())
)

total_ca = CA_par_customer.agg(sum("chiffre_affaire").alias("total")).collect()[0]["total"]


windowSpec = Window.orderBy(desc("chiffre_affaire")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

CA_cumul = (
    CA_par_customer
    .withColumn("ca_cumulé", sum("chiffre_affaire").over(windowSpec))
    .withColumn("ratio_cumulé_en_%", percent_rank().over(windowSpec))
)

top_clients_80 = CA_cumul.filter(col("ratio_cumulé_en_%") <= 80)

top_clients_80.show()



+--------------------+---------------+---------+--------------------+
|  customer_unique_id|chiffre_affaire|ca_cumulé|   ratio_cumulé_en_%|
+--------------------+---------------+---------+--------------------+
|0a0a92112bd4c708c...|        13440.0|  13440.0|                 0.0|
|da122df9eeddfedc1...|         7388.0|  20828.0|1.071306136441549...|
|763c8b1c9c68a0229...|         7160.0|  27988.0|2.142612272883099...|
|dc4802a71eae9be1d...|         6735.0|  34723.0|3.213918409324648...|
|459bef486812aa252...|         6729.0|  41452.0|4.285224545766198...|
|ff4159b92c40ebe40...|         6499.0|  47951.0|5.356530682207747...|
|4007669dec559734d...|         5934.6|  53885.6|6.427836818649297E-5|
|eebb5dda148d3893c...|         4690.0|  58575.6|7.499142955090846E-5|
|48e1ac109decbb877...|         4590.0|  63165.6|8.570449091532397E-5|
|a229eba70ec1c2abe...|         4400.0|  67565.6|9.641755227973946E-5|
|edde2314c6c30e864...|        4399.87| 71965.47|1.071306136441549...|
|fa562ef24d41361e4..

In [ ]:
#df_orders.printSchema()

In [ ]:

#3-Montant totale par type de payment par moi et par années
df_payement = (
    dfs["order_payments"]
    .join(commandes_clients, on="order_id", how="left")
    .selectExpr("order_id", "payment_type", "payment_value", "price", "Extract(year from order_purchase_timestamp) as annee", "Extract(month from order_purchase_timestamp) as mois")
)

montant_par_payment_type = (
    df_payement
    .groupBy("payment_type", "annee" ,"mois")
    .agg(sum(col("price")).alias("montant_toatal"))
)

montant_par_payment_type.show()

+------------+-----+----+------------------+
|payment_type|annee|mois|    montant_toatal|
+------------+-----+----+------------------+
| credit_card| 2017|  12| 578806.2700000053|
| credit_card| 2017|   5|         381232.29|
|     voucher| 2017|   6|24310.320000000032|
|  debit_card| 2018|   7| 33091.34000000004|
|      boleto| 2017|   4| 60866.41000000011|
|  debit_card| 2017|  10|           4939.12|
|     voucher| 2017|   4|19968.969999999976|
|  debit_card| 2018|   2| 6072.059999999996|
|  debit_card| 2017|   7|1684.8900000000003|
|  debit_card| 2016|  10|209.89000000000001|
|      boleto| 2017|   1|19029.259999999962|
| credit_card| 2017|   7|383076.46999999933|
|     voucher| 2017|  10|31484.490000000074|
|      boleto| 2018|   2| 150915.7999999995|
|      boleto| 2018|   3|153955.37999999922|
|     voucher| 2018|   2| 39074.69000000009|
|     voucher| 2017|   3| 23653.99000000001|
|  debit_card| 2017|   2|1269.2800000000002|
|      boleto| NULL|NULL|              NULL|
| credit_c

In [ ]:
#4- Evolution du volume total de commandes

df_orders_clean  = (
    df_orders_sellers
    .join(dfs["order_payments"], on="order_id", how="left")
    .select(
        "order_id",
        "order_purchase_timestamp",
        "payment_value",
        "price"
    )
  )

monthly_orders = (
    df_orders_clean
    .withColumn("annee", year("order_purchase_timestamp"))
    .withColumn("mois", month("order_purchase_timestamp"))
    .groupBy("annee","mois")
    .agg(round(sum("price"), 2 ).alias("montant_total"),

    countDistinct("order_id").alias("nb_commandes")
    )
)

windowSpec = (
    Window.orderBy("annee", "mois")
)

monthly_lag = (
    monthly_orders
    .withColumn("montant_precedent", lag("montant_total").over(windowSpec))
    .withColumn("nb_precedent",lag("nb_commandes").over(windowSpec))
)

evolution_mensuelle = (
    monthly_lag
    .withColumn("evolution_montant_pct", round(((col("montant_total") - col("montant_precedent"))/ col("montant_precedent")) , 2))
    .withColumn("evolution_nb_pct",round(((col("nb_commandes") - col("nb_precedent"))/ col("nb_precedent")) ,2))
    .orderBy("annee","mois")
).show()

+-----+----+-------------+------------+-----------------+------------+---------------------+----------------+
|annee|mois|montant_total|nb_commandes|montant_precedent|nb_precedent|evolution_montant_pct|evolution_nb_pct|
+-----+----+-------------+------------+-----------------+------------+---------------------+----------------+
| 2016|   9|       134.97|           1|             NULL|        NULL|                 NULL|            NULL|
| 2016|  10|     41679.78|         265|           134.97|           1|               307.81|           264.0|
| 2016|  12|         10.9|           1|         41679.78|         265|                 -1.0|            -1.0|
| 2017|   1|    120037.59|         748|             10.9|           1|             11011.62|           747.0|
| 2017|   2|     244342.3|        1641|        120037.59|         748|                 1.04|            1.19|
| 2017|   3|    381305.59|        2546|         244342.3|        1641|                 0.56|            0.55|
| 2017|   

In [ ]:
#Partitionnement des tables des fait

order_by_sate = (
    approved_delivered_orders
    .join(df_order_items, on="order_id", how="inner")
    .join(df_customer, on="customer_id", how="inner")
)

(
    order_by_sate
    .write
    .mode("overwrite")
    .partitionBy("customer_state")
    .parquet("/content/data/fact_orders")
)

In [ ]:
#verification
!ls /content/data/fact_orders/

'customer_state=AC'  'customer_state=MG'  'customer_state=RO'
'customer_state=AL'  'customer_state=MS'  'customer_state=RR'
'customer_state=AM'  'customer_state=MT'  'customer_state=RS'
'customer_state=AP'  'customer_state=PA'  'customer_state=SC'
'customer_state=BA'  'customer_state=PB'  'customer_state=SE'
'customer_state=CE'  'customer_state=PE'  'customer_state=SP'
'customer_state=DF'  'customer_state=PI'  'customer_state=TO'
'customer_state=ES'  'customer_state=PR'   _SUCCESS
'customer_state=GO'  'customer_state=RJ'
'customer_state=MA'  'customer_state=RN'


In [ ]:
#6-commandes dont le montant est supérieur à 3 fois l'écart-type du montant moyen des commandes.

stats = (
    df_orders_clean
    .agg(
        mean("payment_value").alias("moyenne"),
        stddev("payment_value").alias("ecart_type")
        ).collect()[0]
)
moyenne = stats["moyenne"]
ecart_type = stats["ecart_type"]
seuil = moyenne + 3 * ecart_type

anormal_commande = (
    df_orders_clean
    .filter(
        col("payment_value") > seuil
    )
)

anormal_commande.show()

+--------------------+------------------------+-------------+-------+
|            order_id|order_purchase_timestamp|payment_value|  price|
+--------------------+------------------------+-------------+-------+
|403b97836b0c04a62...|     2018-01-02 19:00:43|      1376.45| 1299.0|
|f04bfdbef5359607d...|     2017-09-13 15:07:45|       998.49| 217.85|
|f04bfdbef5359607d...|     2017-09-13 15:07:45|       998.49|  142.9|
|f04bfdbef5359607d...|     2017-09-13 15:07:45|       998.49|  142.9|
|f04bfdbef5359607d...|     2017-09-13 15:07:45|       998.49| 217.85|
|2b72a32a2c3e93fa2...|     2018-04-18 15:40:36|      1255.43| 1148.0|
|ac64f79a33bfa575f...|     2018-04-01 18:27:01|       1119.6| 1099.0|
|6cb134bb285a64b04...|     2018-07-01 20:54:21|      2031.09| 1999.0|
|da8be3bb62e9bf01e...|     2017-04-06 13:44:08|      2751.24| 2690.0|
|4f7ce3efe568a5e57...|     2018-01-26 13:32:22|      1727.67| 1695.0|
|8ca57f2b32c00588c...|     2018-08-06 18:45:01|      1480.73| 1349.0|
|8a0b2bc5c7a1c5246..

In [ ]:
# 7- les produit_id d'une meme commande dans une colonne de type array

orders_products = (
    df_orders_sellers
    .groupBy("order_id")
    .agg(
        collect_set("product_id").alias("products")
    )
)

orders_products.show()

+--------------------+--------------------+
|            order_id|            products|
+--------------------+--------------------+
|00010242fe8c5a6d1...|[4244733e06e7ecb4...|
|000229ec398224ef6...|[c777355d18b72b67...|
|00042b26cf59d7ce6...|[ac6c3623068f30de...|
|00048cc3ae777c65d...|[ef92defde845ab84...|
|0005f50442cb953dc...|[4535b0e1091c278d...|
|00061f2a7bc09da83...|[d63c1011f49d98b9...|
|00063b381e2406b52...|[f177554ea93259a5...|
|0009792311464db53...|[8cab8abac5915871...|
|000c3e6612759851c...|[b50c950aba0dcead...|
|000f25f4d72195062...|[1c05e0964302b6cf...|
|001021efaa8636c29...|[5d7c23067ed3fc8c...|
|0010b2e5201cc5f1a...|[5a419dbf24a8c971...|
|0011d82c4b53e22e8...|[c389f712c4b4510b...|
|00125cb692d048878...|[1c0c0093a48f13ba...|
|00130c0eee84a3d90...|[89321f94e35fc6d7...|
|00137e170939bba5a...|[672e757f331900b9...|
|0014ae671de39511f...|[23365beed316535b...|
|0017afd5076e074a4...|[fe59a1e006df3ac4...|
|0019c29108428acff...|[28b4eced95a52d9c...|
|001ab0a7578dd66cd...|[0b0172eb0

In [ ]:
#df_products_with_categ.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|           225.0|             16.0|             10.0|            14.0|           Léger|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|          1000.0|             30.0|             18.0|            20.0|           Léger|
|96bd76ec8

In [ ]:
#df_orders_sellers.show()